# CSSE4011 Machine Learning and Edge AI Workshop

### 0. Connect to a runtime and change the runtime type to **"T4 GPU"**

Click the down-arrow button in the top-right corner, as shown in the screenshots below.

> **Important:** Please make sure you select **T4 GPU**, as using a different runtime may lead to much longer execution times.

<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/change_runtime_type.png?raw=true" width="300"> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<img src="https://github.com/YangLi309/CSSE4011-ML-Workshop/blob/main/screenshots/runtime_type.png?raw=true" width="300">

## 1. Install Required Packages and Upload MNIST Data

In [ ]:
!git clone https://github.com/YangLi309/CSSE4011-ML-Workshop.git
%cd CSSE4011-ML-Workshop/

In [ ]:
# Install ONNX for model export
!pip3 install onnx onnxscript onnxruntime torchao


## 2. Introduction

### What is PyTorch?
PyTorch is an open-source deep learning framework known for its flexibility and ease of use, making it ideal for rapid prototyping and research.

### What is ONNX?
ONNX (Open Neural Network Exchange) is an open standard format for representing machine learning models, enabling interoperability between various frameworks and devices.

### Why are these important for Edge AI?
- **PyTorch**: Facilitates easy model development and training.
- **ONNX**: Allows exporting models to a standardized format for deployment across different platforms, including resource-constrained edge devices.

### What You Will Learn in This Workshop
- Load and adapt a pretrained model for a new task.
- Evaluate model performance.
- Prune the model to improve inference time.
- Export the model to ONNX format.

## 3. Load Pretrained AlexNet and Adapte the Model for MNIST 10

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import alexnet, AlexNet_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Current device:", device)

# Pretrained AlexNet with the final layer replaced for MNIST (10 classes)
model = alexnet(weights=AlexNet_Weights.DEFAULT)
model.classifier[6] = nn.Linear(4096, 10)  # For 10 classes

model = model.to(device)
print("Pretrained AlexNet model is loaded.")

## 4. Evaluate the Pretrained AlexNet on MNIST

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

# Transform for MNIST images (which are 28x28 grayscale)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to AlexNet input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load MNIST test data
test_dataset = ImageFolder(root='./data/test', transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Evaluate the model (with modified classifier but before fine-tuning)
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of AlexNet with modified classifier (before fine-tuning) on MNIST test data: {100 * correct / total:.2f}%")
print("Note: This accuracy is expected to be low as the model hasn't been finetuned for MNIST yet")

# Sample predictions
print("\nSample predictions:")
for i in range(min(10, len(all_preds))):
    print(f"True: {all_labels[i]}, Predicted: {all_preds[i]}")

## 5. Freeze All Layers Except Classifier

In [ ]:
# Transfer learning: keep ImageNet features fixed and only train the new MNIST head.
for param in model.features.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

## 6. Load MNIST Train Dataset

In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to AlexNet input size
    transforms.Grayscale(num_output_channels=3),  # Convert to 3 channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Assuming your dataset is stored in "./data/" with subfolders 0/, 1/, ..., 9/
train_dataset = ImageFolder(root='./data/train', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

## 7. Finetune the Classifier on MNIST

### Training epochs and corresponding model accuracy

- **1 epoch:** 96.45% accuracy  
- **5 epochs:** 97.02% accuracy  
- **10 epochs:** 99.30% accuracy  

> **Note:** The default setting is **1 epoch** for time efficiency, but you may increase the number of epochs to achieve better performance.

> **Reference:** Training for **1 epoch** should take approximately **3 minutes**. If it takes significantly longer, check whether your runtime type is set to **CPU** instead of **GPU**.

In [ ]:
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# You can increase the number of epochs if you want to have higher accuracy
num_epochs = 1

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Wrap train_loader with tqdm for progress bar
    train_loader_tqdm = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

    for images, labels in train_loader_tqdm:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update progress bar
        train_loader_tqdm.set_postfix({
            'loss': running_loss/total,
            'acc': 100.*correct/total
        })

    # Print epoch statistics
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    print(f'\nEpoch {epoch+1}/{num_epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%')

## 8. Evaluate the Finetuned Model

In [ ]:
# Evaluate the finetuned model on the test set (same loop structure as pre-training eval).
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy of AlexNet with modified classifier (after fine-tuning) on MNIST test data: {100 * correct / total:.2f}%")

# Sample predictions
print("\nSample predictions:")
for i in range(min(10, len(all_preds))):
    print(f"True: {all_labels[i]}, Predicted: {all_preds[i]}")

## 9. Apply INT8 Quantization

In [ ]:
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig

# Reset model
quantized_model = copy.deepcopy(finetuned_model).to(device).eval()

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Current device:", device)

# Copy the model and cast to bfloat16 or float16 (bfloat16 is usually recommended for FP8 matmuls)
quantized_model = copy.deepcopy(finetuned_model).to(device).eval()

# Create a dummy input matching the model's base dtype
dummy_input = torch.randn(1, 3, 224, 224, device=device)

# Quantize the model in-place to Int8 on GPU
quantize_(quantized_model, Int8DynamicActivationInt8WeightConfig())

# Run the FP8 quantized model
with torch.no_grad():
    fp8_output = quantized_model(dummy_input)
    
print("FP8 quantized model executed successfully on GPU.")
print("Output shape:", fp8_output.shape)

## 10. Performance Comparison: Finetuned vs. Quantized Models
*Note: Both models are loaded and executed on the CPU. This provides a clearer baseline comparison of the raw speed improvements achieved through quantization.*

In [ ]:
import time
import os

# Evaluate a model and return accuracy and FPS
def evaluate_model(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    inference_times = []
    
    # Auto-detect what data type the model is using (float32, float16, bfloat16, etc.)
    model_dtype = next(model.parameters()).dtype

    with torch.no_grad():
        for images, labels in dataloader:
            # Move labels to device
            labels = labels.to(device)
            
            # Move images to device AND cast them to the model's data type
            images = images.to(device=device, dtype=model_dtype)

            # Synchronize for accurate GPU timing
            if device.type == "cuda":
                torch.cuda.synchronize()
            start_time = time.time()

            outputs = model(images)

            if device.type == "cuda":
                torch.cuda.synchronize()
            end_time = time.time()

            inference_time = end_time - start_time
            inference_times.append(inference_time)

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_inference_time = sum(inference_times) / len(inference_times)
    fps = 1.0 / (avg_inference_time / images.size(0))

    return accuracy, fps


# Save model and measure file size
def get_model_size_mb(model, file_name):
    torch.save(model.state_dict(), file_name)
    size_mb = os.path.getsize(file_name) / (1024 * 1024)
    os.remove(file_name)
    return size_mb


# Make sure both models are on cpu
device = torch.device("cpu")
finetuned_model = finetuned_model.to(device).eval()
quantized_model = quantized_model.to(device).eval()

# Evaluate the finetuned model
print("Evaluating finetuned PyTorch model...")
finetuned_accuracy, finetuned_fps = evaluate_model(finetuned_model, test_loader, device)
finetuned_size = get_model_size_mb(finetuned_model, "finetuned_model.pth")

# Evaluate the quantized model
print("Evaluating quantized model...")
quantized_accuracy, quantized_fps = evaluate_model(quantized_model, test_loader, device)
quantized_size = get_model_size_mb(quantized_model, "quantized_model.pth")

# Print comparison table
print("\n----- Model Comparison -----")
print(f"{'Model':<20} {'Accuracy':<15} {'FPS':<12} {'Size (MB)':<12}")
print("-" * 60)
print(f"{'Finetuned':<20} {finetuned_accuracy:.2f}%{'':<10} {finetuned_fps:.2f}{'':<6} {finetuned_size:.2f}")
print(f"{'Quantized':<20} {quantized_accuracy:.2f}%{'':<10} {quantized_fps:.2f}{'':<6} {quantized_size:.2f}")
print("-" * 60)

# Calculate differences
acc_diff = quantized_accuracy - finetuned_accuracy
fps_diff = quantized_fps - finetuned_fps
fps_improvement = (fps_diff / finetuned_fps) * 100
size_diff = quantized_size - finetuned_size
size_reduction = ((finetuned_size - quantized_size) / finetuned_size) * 100

print(f"\nAccuracy change after quantization: {acc_diff:.2f}%")
print(f"FPS improvement after quantization: {fps_improvement:.2f}% ({fps_diff:.2f} FPS)")
print(f"Model size change after quantization: {size_diff:.2f} MB")
print(f"Model size reduction after quantization: {size_reduction:.2f}%")

# Note on performance improvements
print("\nNote: quantization may not always lead to a large speedup for small models or small batch sizes.")
print("The actual performance gain depends on the model structure, input size, hardware, and compiler optimizations.")
print("For larger models and heavier workloads, quantization benefits are often more noticeable.")

## 11. Export Quantized Model to ONNX

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ONNX export needs an example batch with the correct shape (NCHW); random values are fine—only the graph is fixed.
dummy_input = torch.randn(1, 3, 224, 224).to(device)
# eval() disables dropout etc. so the exported graph matches inference.
finetuned_model.to(device).eval()
onnx_program = torch.onnx.export(
    finetuned_model,
    (dummy_input,),
    dynamo=True,
)
onnx_program.save("../alexnet_mnist.onnx")


# Export quantized model
quantized_model = quantized_model.to(device).eval()
onnx_program_quant = torch.onnx.export(
    quantized_model,
    (dummy_input,),
    dynamo=True,
)
onnx_program_quant.save("../alexnet_mnist_quantized.onnx")

12. Verify that the exported ONNX models match their PyTorch counterparts (same inputs, same logits within tolerance).

In [ ]:
# Same random input for both checks: PyTorch logits should match ONNXRuntime for each exported model.
import onnxruntime as ort
import numpy as np

def verify_onnx(torch_model, onnx_path, x, label):
    torch_model.eval()
    with torch.no_grad():
        torch_out = torch_model(x).cpu().numpy()
    sess = ort.InferenceSession(onnx_path)
    in_name = sess.get_inputs()[0].name
    onnx_out = sess.run(None, {in_name: x.cpu().numpy()})[0]
    ok = np.allclose(torch_out, onnx_out, atol=1e-4)
    mad = float(np.max(np.abs(torch_out - onnx_out)))
    print(f"{label}: match={ok}, max abs diff={mad:.6e}")


device = next(finetuned_model.parameters()).device
dummy_input = torch.randn(1, 3, 224, 224, device=device)

verify_onnx(finetuned_model, "../alexnet_mnist.onnx", dummy_input, "Finetuned")
verify_onnx(quantized_model, "../alexnet_mnist_quantized.onnx", dummy_input, "Quantized")

## 12. Recap

In this workshop you adapted pretrained AlexNet to MNIST, finetuned the classifier, compared accuracy and throughput against a pruned variant, exported both networks to ONNX, and checked that ONNX Runtime matches PyTorch on the same random input.

**Exported models** (paths follow your export cells, e.g. `../` relative to this notebook):

- **`alexnet_mnist.onnx`** — finetuned model weights.
- **`alexnet_mnist_quantized.onnx`** — quantized finetuned model weights; use as the default deployment artifact.

You can run these in ONNX Runtime, TensorRT, OpenVINO, or any runtime that supports the ONNX opset you exported with.
